<a href="https://colab.research.google.com/github/shaliya-s/Automatic-data-cleaning-and-EDA/blob/main/data_cleansight_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

new cleaning

In [ ]:

!pip install pandas numpy openpyxl scikit-learn


import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from google.colab import files


uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

target_col = input("Enter the name of your target column (e.g., survived, outcome, etc.): ").strip().lower()


def handle_missing(df, strategy='mean'):
    report = []
    for col in df.columns:
        if col == target_col:
            continue
        if df[col].isnull().sum() > 0:
            if strategy == 'mean' and pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].mean())
                report.append(f"[Missing] Filled {col} with mean")
            elif strategy == 'median' and pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
                report.append(f"[Missing] Filled {col} with median")
            elif strategy == 'mode':
                df[col] = df[col].fillna(df[col].mode()[0])
                report.append(f"[Missing] Filled {col} with mode")
            elif strategy == 'drop':
                df = df.dropna(subset=[col])
                report.append(f"[Missing] Dropped rows with null in {col}")
    return df, report

def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    return df, [f"[Duplicate] Removed {before - after} duplicate rows"]

def standardize_column_names(df):
    df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
    return df, ["[Standardize] Column names cleaned"]

def convert_date_columns(df):
    report = []
    for col in df.columns:
        if col == target_col:
            continue  # 🔄 Exclude target
        if "date" in col.lower() or "time" in col.lower():
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
                if df[col].notnull().sum() > 0:
                    report.append(f"[Date] Converted {col} to datetime")
            except Exception as e:
                report.append(f"[Date] Error converting {col}: {str(e)}")
    return df, report

def remove_outliers(df, aggressive=False):
    report = []
    threshold = 3 if not aggressive else 1.5
    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        if col == target_col:
            continue
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        before = df.shape[0]
        df = df[(df[col] >= (Q1 - threshold * IQR)) & (df[col] <= (Q3 + threshold * IQR))]
        after = df.shape[0]
        if before != after:
            report.append(f"[Outlier] Removed {before - after} outliers from {col}")
    return df, report

def normalize_categories(df):
    report = []
    for col in df.select_dtypes(include='object').columns:
        if col == target_col:
            continue
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col].replace(['unknown', 'n/a', 'na', 'none'], np.nan, inplace=True)
        report.append(f"[Category] Normalized text & cleaned unknowns in {col}")
    return df, report

def scale_data(df, method='standard', exclude_cols=['id', 'zipcode', 'zip', 'age_group']):
    report = []
    scaler = StandardScaler() if method == 'standard' else MinMaxScaler()
    numeric_cols = [col for col in df.select_dtypes(include=['float64', 'int64']).columns
                    if col not in exclude_cols and col != target_col]

    if numeric_cols:
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
        report.append(f"[Scale] Scaled columns using {method} scaler (excluded: {exclude_cols + [target_col]})")
    else:
        report.append("[Scale] No numeric columns available for scaling")

    return df, report

def extract_features_from_column(df, col):
    report = []
    if "address" in col:
        df['state'] = df[col].apply(lambda x: str(x).split(',')[-1].strip().split()[0] if pd.notnull(x) else np.nan)
        df['zip_code'] = df[col].str.extract(r'(\d{5})')
        df.drop(columns=[col], inplace=True)
        report.append(f"[Extract] Created 'state' and 'zip_code' from {col}")
    elif "name" in col:
        df['first_name'] = df[col].apply(lambda x: str(x).split()[0] if pd.notnull(x) else np.nan)
        df['last_name'] = df[col].apply(lambda x: str(x).split()[-1] if pd.notnull(x) else np.nan)
        df.drop(columns=[col], inplace=True)
        report.append(f"[Extract] Created 'first_name' and 'last_name' from {col}")
    elif "email" in col:
        df['email_domain'] = df[col].apply(lambda x: str(x).split('@')[-1] if pd.notnull(x) else np.nan)
        df.drop(columns=[col], inplace=True)
        report.append(f"[Extract] Created 'email_domain' from {col}")
    return df, report

def encode_categoricals(df, method='smart'):
    report = []
    categorical_cols = df.select_dtypes(include='object').columns

    for col in categorical_cols:
        if col == target_col:
            continue
        if any(keyword in col for keyword in ['address', 'name', 'email']):
            df, rep = extract_features_from_column(df, col)
            report += rep
        elif method == 'onehot':
            df = pd.get_dummies(df, columns=[col], drop_first=True)
            report.append(f"[Encode] Applied one-hot encoding to {col}")
        elif method == 'label':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            report.append(f"[Encode] Applied label encoding to {col}")

    return df, report


report_log = []
df, log1 = handle_missing(df, strategy='mean')
df, log2 = remove_duplicates(df)
df, log3 = standardize_column_names(df)
df, log4 = convert_date_columns(df)
df, log5 = remove_outliers(df, aggressive=False)
df, log6 = normalize_categories(df)
df, log7 = scale_data(df, method='standard')
df, log8 = encode_categoricals(df, method='smart')

report_log = log1 + log2 + log3 + log4 + log5 + log6 + log7 + log8


print(f"\nUnique values in target column '{target_col}':", df[target_col].unique())
print(f"Data type of target column '{target_col}':", df[target_col].dtype)


print("\nCleaning Summary Report:")
for line in report_log:
    print("-", line)

cleaned_filename = "cleaned_" + filename
df.to_csv(cleaned_filename, index=False)
print("\nCleaned dataset saved as:", cleaned_filename)
files.download(cleaned_filename)

visualizing


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files


uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)


df.fillna(df.mean(numeric_only=True), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)



def visualize_numeric(df):
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        return

    print("\nNUMERICAL COLUMN VISUALIZATIONS")


    print("Histogram")
    df[numeric_cols].hist(bins=30, figsize=(15, 10))
    plt.suptitle("Histograms of Numeric Features", fontsize=16)
    plt.tight_layout()
    plt.show()


    print("Boxplots")
    for col in numeric_cols:
        plt.figure(figsize=(6, 2))
        sns.boxplot(x=df[col])
        plt.title(f'Boxplot of {col}')
        plt.xlabel(col)
        plt.show()


    print("KDE Plots")
    for col in numeric_cols:
        sns.kdeplot(df[col], fill=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Density')
        plt.show()


    print("Correlation Heatmap")
    plt.figure(figsize=(10, 8))
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title("Correlation Between Numerical Features", fontsize=16)
    plt.show()


    if len(numeric_cols) <= 5:
        print(" Pairplot (Numeric Variables)")
        sns.pairplot(df[numeric_cols])
        plt.suptitle("Pairplot of Numerical Features", fontsize=16)
        plt.show()

def visualize_categorical(df):
    categorical_cols = df.select_dtypes(include='object').columns
    if len(categorical_cols) == 0:
        print("No categorical columns found.")
        return

    print("\nCATEGORICAL COLUMN VISUALIZATIONS")

    for col in categorical_cols:
        if df[col].nunique() > 30:
            print(f"⏭️ Skipping '{col}' (too many categories: {df[col].nunique()})")
            continue


        plt.figure(figsize=(10, 4))
        sns.countplot(x=col, data=df)
        plt.title(f'Countplot of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.show()

        plt.figure(figsize=(6, 6))
        df[col].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
        plt.title(f'Pie Chart of {col}')
        plt.ylabel('')
        plt.show()

def visualize_cat_vs_num(df):
    cat_cols = df.select_dtypes(include='object').columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns

    print("\n CATEGORICAL vs NUMERICAL Visualizations")

    for cat in cat_cols:
        if df[cat].nunique() > 10:
            continue
        for num in num_cols:
            plt.figure(figsize=(8, 4))
            sns.boxplot(x=cat, y=num, data=df)
            plt.title(f'{num} by {cat}')
            plt.xlabel(cat)
            plt.ylabel(num)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

visualize_numeric(df)
visualize_categorical(df)
visualize_cat_vs_num(df)


EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("\n📏 Dataset Dimensions:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")


print("\nDescriptive Statistics:")
print(df.describe(include='all'))

print("\nData Types of Each Column:")
print(df.dtypes)



numerical_cols = df.select_dtypes(include=['float64', 'int64', 'datetime64']).columns

if len(numerical_cols) > 0:
    print("\nPlotting histograms for numerical columns...")
    df[numerical_cols].hist(figsize=(10, 8), bins=20)
    plt.suptitle("Distributions of Numerical Features")
    plt.show()
else:
    print("No numerical columns available for plotting histograms.")

# Correlation heatmap for numerical features (only if numerical columns exist)
if len(numerical_cols) > 0:
    print("\nCorrelation Heatmap:")
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', cbar=True)
    plt.title("Correlation Heatmap")
    plt.show()

# Boxplots to detect any remaining outliers in numerical columns
if len(numerical_cols) > 0:
    print("\nBoxplots of Numerical Features:")
    for col in numerical_cols:
        sns.boxplot(x=df[col])
        plt.title(f"Boxplot of {col}")
        plt.show()

# Categorical variable analysis (only if categorical columns exist)
categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print("\nCountplots of Categorical Features:")
    for col in categorical_cols:
        sns.countplot(x=df[col])
        plt.title(f"Countplot of {col}")
        plt.show()

# Pairplots (optional - for exploring relationships between numerical features)
if len(numerical_cols) > 1:
    print("\nPairplot of Numerical Features:")
    sns.pairplot(df[numerical_cols])
    plt.show()

# Checking for missing values in the dataset (if any)
print("\n Missing Values Check:")
missing_data = df.isnull().sum().sort_values(ascending=False)
print(missing_data[missing_data > 0])

# Summary of the EDA process
print("\n Summary of EDA:")
print(f"Number of numerical features: {len(numerical_cols)}")
print(f"Number of categorical features: {len(categorical_cols)}")
print(f"Total rows and columns: {df.shape}")